In [9]:
# LangSmith에서 Tracing을 하고싶다면 키 변경후 아래의 주석 해제
# 반드시 LangChain import 전에 설정
# import os
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "LangSmith에서 발급받은 키"
# os.environ["LANGSMITH_PROJECT"] = "healthcare"

# ============================================================
# 설정 함수
# ============================================================

def get_config() -> dict:
    """
    RAG 파이프라인 전체에서 사용하는 설정값을 반환합니다.
    환경변수나 외부 설정 파일로 교체하기 쉽도록 함수로 분리합니다.
    DB접속정보는 강의환경정보 시트 참조 : https://docs.google.com/spreadsheets/d/145GlLN4vV90KBLFj8g_xdWbVK-2lfE2qGD8u9keZJk8/edit?usp=sharing
    """
    return {
        "db_host":        "classdb.iranglab.com",
        "db_port":        5432,
        "db_name":        "db18",
        "db_user":        "user18",
        "db_password":    "user1899!",
        "ollama_base_url": "http://localhost:11434",
        "llm_model":      "qwen2.5:3b",
        "embed_model":    "bge-m3",
        "embedding_dim":  1024,
        "collection_name": "healthcare_docs",
        "chunk_size":     800,
        "chunk_overlap":  120,
    }

# ============================================================
# DB 연결 함수
# ============================================================
from sqlalchemy import create_engine, text


def get_db_engine(config: dict):
    """
    설정에서 SQLAlchemy 엔진을 생성하고 연결을 검증한 뒤 반환합니다.
    연결 실패 시 예외를 발생시킵니다.
    """
    url = (
        f"postgresql+psycopg://{config['db_user']}:{config['db_password']}"
        f"@{config['db_host']}:{config['db_port']}/{config['db_name']}"
    )
    engine = create_engine(url, pool_pre_ping=True, pool_recycle=1800)
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print(f"PostgreSQL 연결 성공: {config['db_host']}:{config['db_port']}/{config['db_name']}")
    return engine

#DB 초기화
config = get_config()
engine = get_db_engine(config)

PostgreSQL 연결 성공: classdb.iranglab.com:5432/db18


In [10]:
# ============================================================
# PDF 로드 및 텍스트 정제 함수
# ============================================================
import os
import re
from pathlib import Path
from langchain_pymupdf4llm import PyMuPDF4LLMLoader


# ============================================================
# PDF 추출 텍스트 정제
# ============================================================
def _clean_text(text: str) -> str:
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# ============================================================
# 파일명에서 member_id, member_name 추출
# ============================================================
def extract_member_info_from_filename(pdf_path: str) -> dict:
    """
    파일명 형식:
        id__member_id__이름.pdf
    예:
        id__user_002__이서연.pdf
    반환:
        {
            "member_id": "user_002",
            "member_name": "이서연"
        }
    """

    filename = Path(pdf_path).name
    stem = Path(pdf_path).stem

    # id__ 로 시작하지 않으면 회원 문서가 아니라고 판단
    if not stem.startswith("id__"):
        return {}

    parts = stem.split("__")

    if len(parts) != 3:
        raise ValueError(
            f"파일명 형식 오류: {filename}\n"
            "형식은 id__member_id__이름.pdf 이어야 합니다."
        )

    member_id = parts[1].strip()
    member_name = parts[2].strip()

    if not member_id or not member_name:
        raise ValueError(
            f"member_id 또는 이름을 추출할 수 없습니다: {filename}"
        )

    return {
        "member_id": member_id,
        "member_name": member_name,
    }


# ============================================================
# PDF 1개 로드
# ============================================================
def load_pdf(pdf_path: str) -> list:
    """PDF를 로드하고 텍스트를 정제하여 Document 목록으로 반환합니다."""

    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF 파일이 없습니다: {pdf_path}")

    # 파일명에서 회원 정보 추출
    member_info = extract_member_info_from_filename(pdf_path)

    docs = PyMuPDF4LLMLoader(str(pdf_path)).load()

    for doc in docs:
        # 텍스트 정제
        doc.page_content = _clean_text(doc.page_content)

        # ----------------------------------------------------
        # 기본 metadata 추가
        # ----------------------------------------------------
        doc.metadata["source_file"] = os.path.basename(pdf_path)
        doc.metadata["source_path"] = str(pdf_path)
        

        # ----------------------------------------------------
        # 회원 PDF인 경우 metadata에 회원 정보 추가
        # ----------------------------------------------------
        if member_info:
            doc.metadata["member_id"] = member_info["member_id"]
            doc.metadata["member_name"] = member_info["member_name"]
            doc.metadata["source_type"] = "member_health_document"
        else:
            doc.metadata["member_id"] = None
            doc.metadata["member_name"] = None
            doc.metadata["source_type"] = "common_reference_document"

    print(f"PDF 로드 및 클린징 완료: {pdf_path} ({len(docs)}페이지)")

    if member_info:
        print(
            f"metadata 추가 완료: "
            f"member_id={member_info['member_id']}, "
            f"member_name={member_info['member_name']}"
        )
    else:
        print("metadata 추가 완료: 공통 참고문서")

    return docs


# ============================================================
# 폴더 아래의 모든 PDF 로드
# ============================================================
def load_pdfs_from_folder(folder_path: str, recursive: bool = False) -> list:
    """폴더 아래의 모든 PDF를 로드합니다."""

    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"폴더가 없습니다: {folder_path}")

    if not folder.is_dir():
        raise NotADirectoryError(f"폴더 경로가 아닙니다: {folder_path}")

    if recursive:
        pdf_files = sorted(folder.rglob("*.pdf"))
    else:
        pdf_files = sorted(folder.glob("*.pdf"))

    if not pdf_files:
        print(f"PDF 파일이 없습니다: {folder_path}")
        return []

    all_docs = []

    for pdf_file in pdf_files:
        try:
            docs = load_pdf(str(pdf_file))
            all_docs.extend(docs)

        except Exception as e:
            print(f"PDF 로드 실패: {pdf_file}")
            print(f"오류 내용: {e}")

    print("=" * 60)
    print(f"전체 PDF 파일 수: {len(pdf_files)}개")
    print(f"전체 Document 수: {len(all_docs)}개")
    print("=" * 60)

    return all_docs


# ============================================================
# 테스트
# ============================================================
if __name__ == "__main__":
    PDF_FOLDER = "./docs"

    docs = load_pdfs_from_folder(
        folder_path=PDF_FOLDER,
        recursive=False,
    )

    if docs:
        print("\n첫 번째 Document 미리보기:")
        print(docs[0].page_content[:500])

        print("\n첫 번째 Document metadata:")
        print(docs[0].metadata)

PDF 로드 및 클린징 완료: docs\id__user_001__김민준.pdf (3페이지)
metadata 추가 완료: member_id=user_001, member_name=김민준
PDF 로드 및 클린징 완료: docs\id__user_002__이서연.pdf (3페이지)
metadata 추가 완료: member_id=user_002, member_name=이서연
PDF 로드 및 클린징 완료: docs\id__user_003__박지훈.pdf (3페이지)
metadata 추가 완료: member_id=user_003, member_name=박지훈
PDF 로드 및 클린징 완료: docs\id__user_004__최수빈.pdf (3페이지)
metadata 추가 완료: member_id=user_004, member_name=최수빈
PDF 로드 및 클린징 완료: docs\id__user_005__정하늘.pdf (3페이지)
metadata 추가 완료: member_id=user_005, member_name=정하늘
PDF 로드 및 클린징 완료: docs\id__user_006__한지민.pdf (3페이지)
metadata 추가 완료: member_id=user_006, member_name=한지민
PDF 로드 및 클린징 완료: docs\id__user_007__김도윤.pdf (3페이지)
metadata 추가 완료: member_id=user_007, member_name=김도윤
PDF 로드 및 클린징 완료: docs\id__user_008__이민지.pdf (3페이지)
metadata 추가 완료: member_id=user_008, member_name=이민지
PDF 로드 및 클린징 완료: docs\id__user_009__오성민.pdf (3페이지)
metadata 추가 완료: member_id=user_009, member_n

In [12]:
# ============================================================
# Chunk 분할 및 메타데이터 보강 함수
# ============================================================
import os
from datetime import datetime
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ============================================================
# 문서 Chunk 분할
# ============================================================
# RAG에서는 PDF 전체를 한 번에 임베딩하지 않고,
# 검색하기 좋은 작은 단위의 문서 조각(chunk)으로 나눈다.
#
# chunk_size    : 하나의 chunk 최대 글자 수
# chunk_overlap : 앞뒤 chunk가 겹치는 글자 수
#                 문맥이 끊기는 것을 줄이기 위해 사용
# ============================================================
def split_chunks(docs: list, chunk_size: int = 800, chunk_overlap: int = 120) -> list:
    """
    Document 목록을 지정된 크기의 chunk로 분할합니다.

    chunk_size: 하나의 chunk 최대 글자 수
    chunk_overlap: 앞뒤 chunk 간 겹치는 글자 수 (문맥 단절 방지)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""],
    )
    chunks = text_splitter.split_documents(docs)
    print(f"Chunk 분할 완료: {len(docs)}페이지 → {len(chunks)}개 chunk")
    return chunks

# ============================================================
# Chunk 메타데이터 보강
# ============================================================
# PDF Loader가 만든 기본 metadata에
# RAG 저장/검색에 필요한 정보를 추가한다.
# ============================================================
def enrich_metadata(chunks: list, file_path: str) -> list:
    """
    Chunk 메타데이터에 파일 정보, chunk 인덱스, 로드 시각을 추가합니다.
    """
    file_name = os.path.basename(file_path)
    loaded_at = datetime.now().isoformat()
    for i, chunk in enumerate(chunks):
        chunk.metadata.update({
            "file_name":   file_name,
            "file_path":   file_path,
            "source_type": "pdf",
            "loader":      "PyMuPDFLoader",
            "chunk_index": i,
            "loaded_at":   loaded_at,
        })
    print(f"메타데이터 보강 완료: {len(chunks)}개 chunk")
    return chunks


# ============================================================
# 테스트
# ============================================================
if __name__ == "__main__":
    config = get_config()

    chunks = split_chunks(docs, chunk_size=config["chunk_size"], chunk_overlap=config["chunk_overlap"])
    chunks = enrich_metadata(chunks, PDF_FOLDER)

    print(f"\n첫 번째 chunk 메타데이터:")
    print(chunks[0].metadata)

    print(f"\n처음 3개 chunk 미리보기:")
    for i, chunk in enumerate(chunks[:3]):
        print("=" * 60)
        print(f"Chunk {i}: {chunk.page_content[:200]}")

Chunk 분할 완료: 30페이지 → 68개 chunk
메타데이터 보강 완료: 68개 chunk

첫 번째 chunk 메타데이터:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-09T13:28:42+00:00', 'source': 'docs\\id__user_001__김민준.pdf', 'file_path': './docs', 'total_pages': 3, 'format': 'PDF 1.4', 'title': '가상 건강검진 및 의사진단 처방전', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-07-09T13:28:42+00:00', 'trapped': '', 'modDate': "D:20260709132842+00'00'", 'creationDate': "D:20260709132842+00'00'", 'page': 0, 'source_file': 'id__user_001__김민준.pdf', 'source_path': 'docs\\id__user_001__김민준.pdf', 'member_id': 'user_001', 'member_name': '김민준', 'source_type': 'pdf', 'file_name': 'docs', 'loader': 'PyMuPDFLoader', 'chunk_index': 0, 'loaded_at': '2026-07-15T11:44:55.269103'}

처음 3개 chunk 미리보기:
Chunk 0: # **가상 건강검진 결과지** Virtual Health Check Report - Sample ### **중요 안내: 본 문서는 앱 화면에 표시된 제한된 생체·체성분 데이터를 기반으로 생성한 가상 샘플입니다. 실제 의료기관의 검진이나 의사

In [13]:
# ============================================================
# Embedding 모델 함수
# ============================================================
# bge-m3 모델을 사용해 문서 chunk를 벡터로 변환한다.
# bge-m3는 일반적으로 1024차원 벡터를 생성한다.
# ============================================================
from langchain_ollama import OllamaEmbeddings


def create_embedding_model(embed_model: str, ollama_base_url: str):
    """Ollama 임베딩 모델 인스턴스를 생성하여 반환합니다."""
    embeddings = OllamaEmbeddings(
        model=embed_model,
        base_url=ollama_base_url,
    )
    print(f"임베딩 모델 준비 완료: {embed_model}")
    return embeddings


# ============================================================
# Chunk를 임베딩하여 rag_documents 테이블에 저장 함수
# ============================================================
# 처리 흐름:
# 1. chunk 내용 추출
# 2. Ollama bge-m3로 embedding 생성
# 3. vector 차원 검증
# 4. PostgreSQL rag_documents 테이블에 저장
# ============================================================
import json
from sqlalchemy import text as sql_text

# ============================================================
# INSERT SQL
# ============================================================
# SQLAlchemy에서는 %s 대신 :파라미터명 형식을 사용한다.
# pgvector 컬럼에는 문자열을 CAST(:embedding AS vector) 형태로 변환해서 넣는다.
# metadata는 CAST(:metadata AS jsonb) 형태로 변환해서 넣는다.
# ============================================================
_INSERT_SQL = sql_text("""
    INSERT INTO rag_documents (
        source, source_type, page_no, chunk_index,
        content, metadata, embedding
    )
    VALUES (
        :source, :source_type, :page_no, :chunk_index,
        :content, CAST(:metadata AS jsonb), CAST(:embedding AS vector)
    )
""")


def save_chunks_to_db(
    engine,
    chunks: list,
    clear_existing: bool = False,
) -> int:
    """
    Chunk 목록을 임베딩하여 rag_documents 테이블에 저장합니다.

    Returns:
        int: 저장된 chunk 수
    """
    saved_count = 0

    #임베딩 모델 객체 생성
    config = get_config()
    embeddings = create_embedding_model(config["embed_model"], config["ollama_base_url"])

    with engine.begin() as conn:
        if clear_existing:
            conn.execute(sql_text("DELETE FROM rag_documents;"))
            print("기존 데이터 삭제 완료")

        for idx, chunk in enumerate(chunks):
            content  = chunk.page_content
            metadata = chunk.metadata or {}

            source       = metadata.get("source")
            source_type  = metadata.get("source_type", "pdf")
            page_no      = metadata.get("page")
            chunk_index  = metadata.get("chunk_index")

            #chunk 내용을 백터로 임베딩
            vector = embeddings.embed_query(content)

            if len(vector) != config["embedding_dim"]:
                raise ValueError(
                    f"임베딩 차원 불일치: expected={config['embedding_dim']}, actual={len(vector)}"
                )

            vector_text = "[" + ",".join(map(str, vector)) + "]"

            conn.execute(_INSERT_SQL, {
                "source":      source,
                "source_type": source_type,
                "page_no":     page_no,
                "chunk_index": chunk_index,
                "content":     content,
                "metadata":    json.dumps(metadata, ensure_ascii=False),
                "embedding":   vector_text,
            })

            saved_count += 1
            if saved_count % 10 == 0:
                print(f"{saved_count}개 저장 완료")

    print(f"전체 저장 완료: {saved_count}개 chunk")
    return saved_count


# ============================================================
# 테스트
# ============================================================
if __name__ == "__main__":
    config = get_config()
    
    saved = save_chunks_to_db(
        engine=engine,
        chunks=chunks,
        clear_existing=True,
    )
    print(f"\n저장된 chunk 수: {saved}")

임베딩 모델 준비 완료: bge-m3
기존 데이터 삭제 완료
10개 저장 완료
20개 저장 완료
30개 저장 완료
40개 저장 완료
50개 저장 완료
60개 저장 완료
전체 저장 완료: 68개 chunk

저장된 chunk 수: 68
